## Tahap 1 – Penyimpanan Terdistribusi dan Ingesti (HDFS)

### Tujuan
1. Membangun direktori terdistribusi di HDFSuntuk memisahkan  data mentah, data aliran, data bersih, dan wadah untuk model machine learning.
2. Mengunggah seluruh dataset multi-format (CSV transaksi batch, JSON Lines pelanggan dan produk, dan CSV micro-batch streaming) ke HDFS agar bisa digunakan sebagai sumber data pada tahap selanjutnya. seluruh tahap pemrosesan selanjutnya membaca data langsung dari `hdfs://localhost:9000/tidarmart/...` dan bukan dari path lokal.
3. Memverifikasi keberhasilan upload data serta mendokumentasikan bukti karakteristik **Volume** melalui pemeriksaan ukuran berkas , jumlah blok, status kesehatan, dan faktor replikasi.

In [2]:
%%bash
#membuat direktori yang diperluakan di HDFS
hdfs dfs -mkdir -p /tidarmart/raw/transaksi
hdfs dfs -mkdir -p /tidarmart/raw/pelanggan
hdfs dfs -mkdir -p /tidarmart/raw/produk
hdfs dfs -mkdir -p /tidarmart/stream_input
hdfs dfs -mkdir -p /tidarmart/curated
hdfs dfs -mkdir -p /tidarmart/models

# 2. Unggah data ke HDFS (-f agar menimpa jika dijalankan ulang)
hdfs dfs -put -f data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
hdfs dfs -put -f data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
hdfs dfs -put -f data_proyek/produk.json /tidarmart/raw/produk/
hdfs dfs -put -f data_proyek/stream_input/*.csv /tidarmart/stream_input/

# 3. Bukti ukuran, jumlah blok, dan faktor replikasi
hdfs dfs -du -h /tidarmart/raw
hdfs dfs -du -h /tidarmart/stream_input
hdfs fsck /tidarmart -files -blocks

6.0 M    6.0 M    /tidarmart/raw/pelanggan
684.3 K  684.3 K  /tidarmart/raw/produk
187.2 M  187.2 M  /tidarmart/raw/transaksi
479.9 K  479.9 K  /tidarmart/stream_input/stream_01.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_02.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_03.csv
479.2 K  479.2 K  /tidarmart/stream_input/stream_04.csv
479.7 K  479.7 K  /tidarmart/stream_input/stream_05.csv
479.0 K  479.0 K  /tidarmart/stream_input/stream_06.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_07.csv
479.1 K  479.1 K  /tidarmart/stream_input/stream_08.csv
479.3 K  479.3 K  /tidarmart/stream_input/stream_09.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_10.csv
479 K    479 K    /tidarmart/stream_input/stream_11.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_12.csv
479.6 K  479.6 K  /tidarmart/stream_input/stream_13.csv
479.5 K  479.5 K  /tidarmart/stream_input/stream_14.csv
479.8 K  479.8 K  /tidarmart/stream_input/stream_15.csv
479.0 K  479.0 K  /tidarmart/strea

Connecting to namenode via http://localhost:9870/fsck?ugi=cahyoalim&files=1&blocks=1&path=%2Ftidarmart


FSCK started by cahyoalim (auth:SIMPLE) from /127.0.0.1 for path /tidarmart at Mon Nov 23 11:42:44 WIB 2026

/tidarmart <dir>
/tidarmart/curated <dir>
/tidarmart/models <dir>
/tidarmart/raw <dir>
/tidarmart/raw/pelanggan <dir>
/tidarmart/raw/pelanggan/pelanggan.json 6307387 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1724578924-127.0.1.1-1788753323439:blk_1073741919_1101 len=6307387 Live_repl=1

/tidarmart/raw/produk <dir>
/tidarmart/raw/produk/produk.json 700710 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1724578924-127.0.1.1-1788753323439:blk_1073741920_1102 len=700710 Live_repl=1

/tidarmart/raw/transaksi <dir>
/tidarmart/raw/transaksi/transaksi_part1.csv 49074339 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1724578924-127.0.1.1-1788753323439:blk_1073741915_1097 len=49074339 Live_repl=1

/tidarmart/raw/transaksi/transaksi_part2.csv 49077730 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1724578924-127.0.1.1-1788753323439:blk_1073741916_109

### Interpretasi Tahap 1

**1. Upload Data & Bukti Karakteristik Volume**
* Seluruh dataset berhasil diunggah ke HDFS dengan status **`HEALTHY`** (`Missing blocks: 0`, `Corrupt blocks: 0`). Total terdapat **26 berkas** yang tersimpan dalam **26 blok** tervalidasi dengan total ukuran **sekitar 203,3 MiB**.
* Rincian kapasitas penyimpanan pada raw dan stream_input:
  * **Transaksi batch : 187,2 MB** (4 berkas CSV, masing-masing sekitar 49 MB, total 2.010.000 baris)
  * **Pelanggan : 6,0 MB** (50.000 dokumen JSON Lines).
  * **Produk : 684,3 KB** (5.000 dokumen JSON Lines).
  * **Micro-batch streaming :** 20 berkas CSV, masing-masing berukuran sekitar 479–480 KB.
* Setiap berkas menempati tepat **1 blok** karena ukuran berkas terbesar (~49 MB) masih berada di bawah batas ukuran blok bawaan HDFS (128 MB).

**2. Alasan Pemisahan Direktori `raw` dan `curated`**
* Direktori `raw` berfungsi sebagai zona penyimpanan data mentah yang bersifat immutable dan single source of truth. Data tidak pernah diubah langsung di folder ini agar jika terjadi kesalahan pembersihan atau perubahan kebutuhan bisnis di masa depan, pipeline ETL tetap dapat dijalankan ulang dari data asli tanpa kehilangan informasi.
* Direktori `curated` khusus menyimpan data hasil pembersihan, validasi, dan transformasi dalam format kolumnar atau Parquet yang siap digunakan secara cepat dan konsisten untuk analisis maupun model machine learning.

**3. Makna Faktor Replikasi = 1 pada Lingkungan *Pseudo-Distributed* vs. Klaster Nyata**
* Hasil `hdfs fsck` menunjukkan `Number of data-nodes: 1` dan `Default replication factor: 1` (`Average block replication: 1.0`), sehingga kolom ukuran asli dan ukuran terpakai pada `hdfs dfs -du -h` bernilai identik (misalnya `187.2 M` dan `187.2 M`).
* Pada lingkungan lokal (*pseudo-distributed*), seluruh layanan Hadoop berjalan pada satu mesin saja sehingga replikasi 1 sudah memadai untuk simulasi dan menghemat ruang penyimpanan, namun **tidak memiliki toleransi kesalahan (*fault tolerance*)**—jika penyimpanan lokal rusak, data tidak memiliki salinan cadangan.
* Sebaliknya, pada klaster produksi nyata yang terdiri atas banyak server, faktor replikasi umumnya ditetapkan sebesar **3** dan disebar ke beberapa *DataNode* serta rak berbeda. Skema ini menjamin ketersediaan tinggi bila ada node yang mati.

## Tahap 2 – Profiling Kualitas Data (Veracity & Variety)

### Tujuan
1. Menginisialisasi `SparkSession` sesuai batasan sumber daya proyek (`local[2]`, memori driver `1536m`, `shuffle.partitions = 4`, dan AQE aktif) serta membaca seluruh dataset langsung dari HDFS menggunakan skema eksplisit (`StructType`).
2. Melakukan audit kualitas data (*Veracity*) pada data mentah transaksi mencakup deteksi duplikat `order_id`, persentase nilai kosong per kolom, harga negatif, *outlier* kuantitas berbasis kuantil, dan ketidakkonsistenan penulisan nama kota.
3. Menyajikan hasil audit dalam satu tabel ringkasan **"Sebelum Pembersihan"** sebagai acuan aturan ETL pada Tahap 3.
4. Memeriksa profil data `pelanggan` dan `produk` untuk membuktikan karakteristik **Variety** (atribut produk yang bervariasi per kategori serta penanganan field array opsional `tags`).

In [3]:
from pyspark.sql import SparkSession

#membuat sparksession
spark = (SparkSession.builder
    .appName('TidarMart')
    .master('local[2]')
    .config('spark.driver.memory', '1536m')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

print(spark.version)

3.5.9


In [4]:
print("Menentukan skema dan membaca data dari HDFS")

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, ArrayType
)

#menentukan path sumber data
sumber_data = "hdfs://localhost:9000/tidarmart"

#menentukan skema
skema_transaksi = StructType([
    StructField("order_id", StringType(), True),
    StructField("tanggal", TimestampType(), True),
    StructField("pelanggan_id", IntegerType(), True),
    StructField("produk_id", IntegerType(), True),
    StructField("qty", DoubleType(), True),
    StructField("harga_satuan", DoubleType(), True),
    StructField("diskon_persen", IntegerType(), True),
    StructField("metode_pembayaran", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("berat_kg", DoubleType(), True),
    StructField("jarak_km", DoubleType(), True),
    StructField("lama_kirim_hari", IntegerType(), True),
    StructField("status_pesanan", StringType(), True),  
])

skema_pelanggan = StructType([
    StructField("pelanggan_id", IntegerType(), True),
    StructField("nama", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("tanggal_daftar", StringType(), True),
    StructField("membership", StringType(), True),
])

skema_produk = StructType([
    StructField("produk_id", IntegerType(), True),
    StructField("nama_produk", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("harga_dasar", IntegerType(), True),
    StructField("berat_kg", DoubleType(), True),
    StructField("garansi_bulan", IntegerType(), True),
    StructField("voltase", IntegerType(), True),
    StructField("ukuran", StringType(), True),
    StructField("bahan", StringType(), True),
    StructField("kadaluarsa_hari", IntegerType(), True),
    StructField("penerbit", StringType(), True),
    StructField("tags", ArrayType(StringType()), True),
])

#Membaca data mentah langsung dari HDFS
transaksi_mentah = spark.read.csv(f"{sumber_data}/raw/transaksi/", header=True, schema=skema_transaksi)
pelanggan_mentah = spark.read.json(f"{sumber_data}/raw/pelanggan/pelanggan.json", schema=skema_pelanggan)
produk_mentah = spark.read.json(f"{sumber_data}/raw/produk/produk.json", schema=skema_produk)

transaksi_mentah.printSchema()
transaksi_mentah.show(5, truncate=False)

Menentukan skema dan membaca data dari HDFS
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- pelanggan_id: integer (nullable = true)
 |-- produk_id: integer (nullable = true)
 |-- qty: double (nullable = true)
 |-- harga_satuan: double (nullable = true)
 |-- diskon_persen: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- berat_kg: double (nullable = true)
 |-- jarak_km: double (nullable = true)
 |-- lama_kirim_hari: integer (nullable = true)
 |-- status_pesanan: string (nullable = true)



+----------+-------------------+------------+---------+---+------------+-------------+-----------------+----------+--------+--------+---------------+--------------+
|order_id  |tanggal            |pelanggan_id|produk_id|qty|harga_satuan|diskon_persen|metode_pembayaran|kota      |berat_kg|jarak_km|lama_kirim_hari|status_pesanan|
+----------+-------------------+------------+---------+---+------------+-------------+-----------------+----------+--------+--------+---------------+--------------+
|ORD2000000|2026-02-18 01:06:23|29543       |1509     |2.0|15000.0     |0            |COD              |Semarang  |24.22   |83.4    |3              |Selesai       |
|ORD2000001|2026-05-15 04:43:31|6357        |8        |5.0|120000.0    |0            |Transfer Bank    |temanggung|6.45    |55.2    |3              |Selesai       |
|ORD2000002|2026-04-03 13:52:28|28797       |1823     |1.0|85000.0     |0            |COD              |Wonosobo  |6.08    |98.2    |3              |Selesai       |
|ORD200000

In [5]:
print("Kualitas Data Transaksi Sebelum Pembersihan\n")

from pyspark.sql.functions import (
    col, count, sum as spark_sum, when, round,
    trim, initcap, countDistinct, isnan
)

#total baris dan duplikat order_id
total_baris = transaksi_mentah.count()
order_id_unik = transaksi_mentah.select("order_id").distinct().count()
duplikat_order_id = total_baris - order_id_unik



#Hitung nilai kosong di setiap kolom
kolom_desimal = ["qty", "harga_satuan", "berat_kg", "jarak_km"]
daftar_cek_null = []

for nama_kolom in transaksi_mentah.columns:
    if nama_kolom in kolom_desimal:
        kondisi_kosong = col(nama_kolom).isNull() | isnan(col(nama_kolom))
    else:
        kondisi_kosong = col(nama_kolom).isNull()
    daftar_cek_null.append(spark_sum(when(kondisi_kosong, 1).otherwise(0)).alias(nama_kolom))

hasil_null = transaksi_mentah.agg(*daftar_cek_null).collect()[0].asDict()

tabel_null = pd.DataFrame([
    {
        "Kolom": k,
        "Jumlah Null": hasil_null[k],
        "Persentase (%)": __builtins__.round(hasil_null[k] / total_baris * 100, 4)
    }
    for k in transaksi_mentah.columns
])
print("--=Persentase Nilai Kosong per Kolom")
display(tabel_null)



#Distribusi kuantitas (qty) dan batas outlier dengan kuantil (IQR)
kuantil_qty = transaksi_mentah.filter(col("qty").isNotNull()).approxQuantile(
    "qty", [0.0, 0.25, 0.50, 0.75, 0.99, 0.999, 1.0], 0.0001
)
q_min, q1, median_qty, q3, p99, p999, q_max = kuantil_qty
iqr = q3 - q1
batas_atas_outlier = q3 + 1.5 * iqr

print("\n--=Distribusi Kuantil qty")
print(f"Min={q_min}, Q1={q1}, Median={median_qty}, Q3={q3}, P99={p99}, P99.9={p999}, Max={q_max}")
print(f"IQR = {iqr} | Batas Atas Outlier (Q3 + 1.5*IQR) = {batas_atas_outlier}")



#menghitung harga negatif, outlier qty, dan inkonsistensi penulisan kota
anomali = transaksi_mentah.agg(
    spark_sum(when(col("harga_satuan") < 0, 1).otherwise(0)).alias("harga_negatif"),
    spark_sum(when(col("qty") > batas_atas_outlier, 1).otherwise(0)).alias("qty_outlier"),
    spark_sum(when(col("kota") != initcap(trim(col("kota"))), 1).otherwise(0)).alias("kota_berantakan"),
    countDistinct("kota").alias("jumlah_variasi_kota")
).collect()[0]

print("\n--=Variasi Penulisan Nama Kota")
transaksi_mentah.groupBy("kota").count().orderBy("kota").show(30, truncate=False)



#menyimpan dan menampilkan tabel ringkasan "Sebelum Pembersihan" 
profil_sebelum = {
    "Total Baris": total_baris,
    "Order ID Unik": order_id_unik,
    "Duplikat Order ID": duplikat_order_id,
    "Null pada qty": hasil_null["qty"],
    "Null pada harga_satuan": hasil_null["harga_satuan"],
    "Harga Satuan Negatif (< 0)": anomali["harga_negatif"],
    f"Outlier qty (> {batas_atas_outlier:.1f})": anomali["qty_outlier"],
    "Penulisan Kota Tidak Konsisten": anomali["kota_berantakan"],
    "Jumlah Variasi String Kota": anomali["jumlah_variasi_kota"],
}

tabel_sebelum_pembersihan = pd.DataFrame([
    {
        "Indikator Kualitas Data": indikator,
        "Sebelum Pembersihan": f"{nilai:,}",
        "Persentase (%)": f"{(nilai / total_baris * 100):.3f}%"
        if indikator not in ["Total Baris", "Jumlah Variasi String Kota"] else "-"
    }
    for indikator, nilai in profil_sebelum.items()
])

print("--=Tabel Ringkasan Sebelum Pembersihan")
display(tabel_sebelum_pembersihan)

Kualitas Data Transaksi Sebelum Pembersihan



[Stage 10:>                                                         (0 + 2) / 2]

--=Persentase Nilai Kosong per Kolom


,Kolom,Jumlah Null,Persentase (%)
0,order_id,0,0.0000
1,tanggal,0,0.0000
2,pelanggan_id,0,0.0000
3,produk_id,0,0.0000
4,qty,6173,0.3071
5,harga_satuan,5930,0.2950
6,diskon_persen,0,0.0000
7,metode_pembayaran,0,0.0000
8,kota,0,0.0000
9,berat_kg,0,0.0000



--=Distribusi Kuantil qty
Min=1.0, Q1=2.0, Median=4.0, Q3=6.0, P99=7.0, P99.9=7.0, Max=999.0
IQR = 4.0 | Batas Atas Outlier (Q3 + 1.5*IQR) = 12.0



--=Variasi Penulisan Nama Kota


[Stage 20:>                                                         (0 + 2) / 2]

+----------+------+
|kota      |count |
+----------+------+
|KEBUMEN   |6219  |
|Kebumen   |238424|
|MAGELANG  |6289  |
|Magelang  |239168|
|PURWOREJO |6245  |
|Purworejo |239582|
|SEMARANG  |6162  |
|SOLO      |6396  |
|Semarang  |238629|
|Solo      |237696|
|TEMANGGUNG|6277  |
|Temanggung|238587|
|WONOSOBO  |6224  |
|Wonosobo  |238739|
|YOGYAKARTA|6244  |
|Yogyakarta|238407|
|kebumen   |6250  |
|magelang  |6334  |
|purworejo |6266  |
|semarang  |6503  |
|solo      |6416  |
|temanggung|6307  |
|wonosobo  |6385  |
|yogyakarta|6251  |
+----------+------+

--=Tabel Ringkasan Sebelum Pembersihan


,Indikator Kualitas Data,Sebelum Pembersihan,Persentase (%)
0,Total Baris,"2,010,000",-
1,Order ID Unik,"2,000,000",99.502%
2,Duplikat Order ID,"10,000",0.498%
3,Null pada qty,"6,173",0.307%
4,Null pada harga_satuan,"5,930",0.295%
5,Harga Satuan Negatif (< 0),"2,040",0.101%
6,Outlier qty (> 12.0),969,0.048%
7,Penulisan Kota Tidak Konsisten,"100,768",5.013%
8,Jumlah Variasi String Kota,24,-


In [6]:
from pyspark.sql.functions import avg, coalesce, array, size

#Profil Data Pelanggan
total_pelanggan = pelanggan_mentah.count()
pelanggan_unik = pelanggan_mentah.select("pelanggan_id").distinct().count()
print(f"--=Profil Pelanggan (Total: {total_pelanggan:,} | Unik: {pelanggan_unik:,})")
pelanggan_mentah.groupBy("membership").count().orderBy(col("count").desc()).show()



#Profil Produk & Persentase Atribut per Kategori
total_produk = produk_mentah.count()
atribut_opsional = ["garansi_bulan", "voltase", "ukuran", "bahan", "kadaluarsa_hari", "penerbit", "tags"]

daftar_agregasi = [count("*").alias("jumlah_produk")]
for atribut in atribut_opsional:
    persen_ada = round(avg(when(col(atribut).isNotNull(), 1).otherwise(0)) * 100, 1)
    daftar_agregasi.append(persen_ada.alias(atribut))

profil_kategori = produk_mentah.groupBy("kategori").agg(*daftar_agregasi).orderBy("kategori")

print(f"--=Persentase Atribut Produk per Kategori (%) dari {total_produk:,} Produk")
display(profil_kategori.toPandas())



#Pemeriksaan dan contoh penanganan field array opsional 'tags'
produk_tags = produk_mentah.filter(col("tags").isNotNull()).count()
print(f"Produk dengan 'tags': {produk_tags:,} | Tanpa 'tags' (null): {total_produk - produk_tags:,}")

# Ubah null pada 'tags' menjadi array kosong [] agar aman saat dihitung ukurannya atau di-explode
produk_tags_aman = produk_mentah.select(
    "produk_id",
    "kategori",
    "tags",
    coalesce(col("tags"), array().cast(ArrayType(StringType()))).alias("tags_bersih"),
    size(coalesce(col("tags"), array().cast(ArrayType(StringType())))).alias("jumlah_tag")
)
produk_tags_aman.show(8, truncate=False)

--=Profil Pelanggan (Total: 50,000 | Unik: 50,000)
+----------+-----+
|membership|count|
+----------+-----+
|    Silver|30060|
|      Gold|15075|
|  Platinum| 4865|
+----------+-----+

--=Persentase Atribut Produk per Kategori (%) dari 5,000 Produk


,kategori,jumlah_produk,garansi_bulan,voltase,ukuran,bahan,kadaluarsa_hari,penerbit,tags
0,Buku,584,0.0,0.0,0.0,0.0,0.0,100.0,31.0
1,Elektronik,635,100.0,100.0,0.0,0.0,0.0,0.0,30.9
2,Fashion,624,0.0,0.0,100.0,100.0,0.0,0.0,30.9
3,Kesehatan,668,0.0,0.0,0.0,0.0,0.0,0.0,31.1
4,Makanan,647,0.0,0.0,0.0,0.0,100.0,0.0,29.7
5,Olahraga,576,0.0,0.0,0.0,0.0,0.0,0.0,29.2
6,Otomotif,593,0.0,0.0,0.0,0.0,0.0,0.0,32.9
7,Rumah Tangga,673,0.0,0.0,0.0,0.0,0.0,0.0,28.8


Produk dengan 'tags': 1,527 | Tanpa 'tags' (null): 3,473
+---------+------------+--------------+--------------+----------+
|produk_id|kategori    |tags          |tags_bersih   |jumlah_tag|
+---------+------------+--------------+--------------+----------+
|1        |Elektronik  |NULL          |[]            |0         |
|2        |Kesehatan   |NULL          |[]            |0         |
|3        |Kesehatan   |[impor, lokal]|[impor, lokal]|2         |
|4        |Fashion     |NULL          |[]            |0         |
|5        |Rumah Tangga|NULL          |[]            |0         |
|6        |Makanan     |NULL          |[]            |0         |
|7        |Otomotif    |[lokal, impor]|[lokal, impor]|2         |
|8        |Buku        |NULL          |[]            |0         |
+---------+------------+--------------+--------------+----------+
only showing top 8 rows



### Interpretasi Tahap 2

**1. Audit Kualitas Data Transaksi (*Veracity*)**
* Dari total **2.010.000 baris** data mentah transaksi, hanya **2.000.000** yang merupakan `order_id` unik; ditemukan **10.000 baris duplikat (0,498%)** yang berpotensi menyebabkan perhitungan ganda (*double counting*) pada total pendapatan.
* Nilai kosong (*null*) terisolasi hanya pada dua kolom numerik utama, yaitu `qty` sebanyak **6.173 baris (0,307%)** dan `harga_satuan` sebanyak **5.930 baris (0,295%)**, sementara 11 kolom lainnya lengkap (0% *null*).
* Ditemukan **2.040 baris (0,101%)** dengan `harga_satuan` bernilai negatif (`< 0`) yang tidak valid secara bisnis.
* Analisis kuantil pada kolom `qty` menunjukkan bahwa mayoritas transaksi normal berada pada rentang **1 hingga 7 unit** ($Q_1 = 2$, Median $= 4$, $Q_3 = 6$, dan $P_{99.9} = 7$), namun terdapat lonjakan ekstrem hingga `Max = 999`. Menggunakan batas atas metode IQR ($Q_3 + 1{,}5 \times \text{IQR} = 12{,}0$), terdeteksi **969 transaksi (0,048%)** sebagai *outlier* kuantitas.
* Penulisan kolom `kota` tidak seragam pada **100.768 baris (5,013%)** akibat perbedaan kapitalisasi (misalnya `MAGELANG`, `Magelang`, dan `magelang`), sehingga 8 kota operasional terbaca sebagai **24 string kota berbeda**. Seluruh anomali ini menjadi dasar aturan pembersihan pada Tahap 3.

**2. Profil Pelanggan & Variasi Skema Produk (*Variety*)**
* Dataset `pelanggan` berisi **50.000 baris** yang seluruhnya unik (`50.000` `pelanggan_id`) dengan distribusi segmen *membership* didominasi oleh **Silver (30.060)**, diikuti **Gold (15.075)**, dan **Platinum (4.865)**.
* Dataset `produk` (**5.000 dokumen**) membuktikan karakteristik *Variety* pada format semi-terstruktur JSON: atribut tambahan muncul secara spesifik berdasarkan kategori produk—yakni `penerbit` (100% pada **Buku**), `garansi_bulan` dan `voltase` (100% pada **Elektronik**), `ukuran` dan `bahan` (100% pada **Fashion**), serta `kadaluarsa_hari` (100% pada **Makanan**).
* Field array opsional `tags` hanya dimiliki oleh **1.527 produk (30,54%)**, sedangkan **3.473 produk (69,46%)** bernilai `null`. Untuk mencegah error `size() = -1` atau hilangnya baris saat operasi *array*, nilai `null` pada `tags` ditangani menggunakan fungsi `coalesce(..., array())` sehingga dikonversi aman menjadi array kosong `[]` dengan panjang `0`.

## Tahap 3 – ETL dan Penyimpanan Kolumnar (Parquet)

### Tujuan
1. Membersihkan dataset transaksi berdasarkan temuan audit Tahap 2 (deduplikasi `order_id`, penanganan nilai kosong, harga negatif, *outlier* `qty`, dan normalisasi nama `kota`) disertai justifikasi teknis untuk setiap aturan pembersihan.
2. Menampilkan laporan perbandingan **"Sebelum vs. Sesudah Pembersihan"** secara berdampingan untuk memverifikasi seluruh anomali telah tertangani.
3. Menambahkan empat kolom turunan wajib: `total_nilai`, `bulan`, `hari_dalam_minggu`, dan `flag_batal`.
4. Menyimpan hasil bersih ke `/tidarmart/curated/transaksi` di HDFS dalam format **Parquet** berpartisi kolom `bulan`, lalu membandingkan ukuran penyimpanan dan waktu eksekusi satu query agregasi yang sama antara CSV mentah dan Parquet.

In [7]:
from pyspark.sql.functions import (
    col, sum as spark_sum, when, round, trim, initcap, lower,
    countDistinct, isnan, date_format, dayofweek
)
from pyspark.sql.types import IntegerType

print("PEMBERSIHAN DATA\n")


# 1. Pembersihan Data Transaksi beserta alasan tiap aturan:
# - Duplikat order_id -> DIBUANG (dropDuplicates): ID transaksi harus unik agar omzet tidak terhitung ganda.
# - Null pada qty & harga_satuan -> DIBUANG (filter isNotNull): kolom inti nilai transaksi; imputasi rata-rata berisiko mendistorsi nilai pesanan aktual.
# - Harga satuan <= 0 -> DIBUANG (filter > 0): harga negatif/nol tidak valid secara bisnis (kesalahan sistem input).
# - Outlier qty (> 12.0) -> DIBUANG (filter <= batas_atas_outlier): 99,9% transaksi normal bernilai 1-7 unit; nilai ekstrem hingga 999 merupakan anomali input.
# - Penulisan kota tidak seragam -> DIKOREKSI (initcap + trim): hanya berbeda kapitalisasi/spasi, sehingga dapat diseragamkan tanpa kehilangan baris transaksi.

transaksi_bersih = (
    transaksi_mentah
    .dropDuplicates(["order_id"])
    .filter(
        col("qty").isNotNull() & ~isnan(col("qty")) &
        (col("qty") >= 1) & (col("qty") <= batas_atas_outlier)
    )
    .filter(
        col("harga_satuan").isNotNull() & ~isnan(col("harga_satuan")) &
        (col("harga_satuan") > 0)
    )
    .withColumn("kota", initcap(trim(col("kota"))))
    .withColumn("qty", col("qty").cast(IntegerType()))
)



# 2. Tambahkan 4 kolom turunan wajib sesuai instruksi
transaksi_curated = (
    transaksi_bersih
    .withColumn("total_nilai", round(col("qty") * col("harga_satuan") * (1 - col("diskon_persen") / 100.0), 2))
    .withColumn("bulan", date_format(col("tanggal"), "yyyy-MM"))
    .withColumn("hari_dalam_minggu", dayofweek(col("tanggal")))  # 1 = Minggu, 2 = Senin, ..., 7 = Sabtu
    .withColumn("flag_batal", when(lower(col("status_pesanan")).contains("batal"), 1).otherwise(0))
)

# Cek hasil pembuatan kolom turunan & verifikasi pemetaan flag_batal
print("--Flag Batal")
transaksi_curated.groupBy("status_pesanan", "flag_batal").count().show()

print("--==data curated")
transaksi_curated.select(
    "order_id", "tanggal", "bulan", "hari_dalam_minggu",
    "qty", "harga_satuan", "diskon_persen", "total_nilai",
    "status_pesanan", "flag_batal"
).show(5, truncate=False)



# 3. Laporan "Sesudah Pembersihan" berdampingan dengan "Sebelum Pembersihan"
total_sesudah = transaksi_curated.count()
unik_sesudah = transaksi_curated.select("order_id").distinct().count()

cek_sesudah = transaksi_curated.agg(
    spark_sum(when(col("qty").isNull(), 1).otherwise(0)).alias("null_qty"),
    spark_sum(when(col("harga_satuan").isNull(), 1).otherwise(0)).alias("null_harga"),
    spark_sum(when(col("harga_satuan") < 0, 1).otherwise(0)).alias("harga_negatif"),
    spark_sum(when(col("qty") > batas_atas_outlier, 1).otherwise(0)).alias("qty_outlier"),
    spark_sum(when(col("kota") != initcap(trim(col("kota"))), 1).otherwise(0)).alias("kota_berantakan"),
    countDistinct("kota").alias("jumlah_variasi_kota")
).collect()[0]

profil_sesudah = {
    "Total Baris": total_sesudah,
    "Order ID Unik": unik_sesudah,
    "Duplikat Order ID": total_sesudah - unik_sesudah,
    "Null pada qty": cek_sesudah["null_qty"],
    "Null pada harga_satuan": cek_sesudah["null_harga"],
    "Harga Satuan Negatif (< 0)": cek_sesudah["harga_negatif"],
    f"Outlier qty (> {batas_atas_outlier:.1f})": cek_sesudah["qty_outlier"],
    "Penulisan Kota Tidak Konsisten": cek_sesudah["kota_berantakan"],
    "Jumlah Variasi String Kota": cek_sesudah["jumlah_variasi_kota"],
}

tabel_perbandingan = pd.DataFrame([
    {
        "Indikator Kualitas Data": k,
        "Sebelum Pembersihan": f"{profil_sebelum[k]:,}",
        "Sesudah Pembersihan": f"{profil_sesudah[k]:,}",
    }
    for k in profil_sebelum.keys()
])

print("--=Tabel Perbandingan Sebelum vs. Sesudah Pembersihan")
display(tabel_perbandingan)

PEMBERSIHAN DATA

--Flag Batal


+--------------+----------+-------+
|status_pesanan|flag_batal|  count|
+--------------+----------+-------+
|       Selesai|         0|1533001|
|    Dibatalkan|         1| 392180|
|  Dikembalikan|         0|  59813|
+--------------+----------+-------+

--==data curated


+----------+-------------------+-------+-----------------+---+------------+-------------+-----------+--------------+----------+
|order_id  |tanggal            |bulan  |hari_dalam_minggu|qty|harga_satuan|diskon_persen|total_nilai|status_pesanan|flag_batal|
+----------+-------------------+-------+-----------------+---+------------+-------------+-----------+--------------+----------+
|ORD1000000|2026-06-04 10:47:18|2026-06|5                |6  |350000.0    |5            |1995000.0  |Selesai       |0         |
|ORD1000009|2026-04-27 03:57:08|2026-04|2                |5  |25000.0     |15           |106250.0   |Dibatalkan    |1         |
|ORD1000010|2026-04-10 02:41:14|2026-04|6                |7  |25000.0     |5            |166250.0   |Selesai       |0         |
|ORD1000013|2026-01-16 08:22:31|2026-01|6                |2  |1500000.0   |15           |2550000.0  |Selesai       |0         |
|ORD1000016|2026-05-15 21:40:32|2026-05|6                |5  |1500000.0   |0            |7500000.0  |Sel

[Stage 73:>                                                         (0 + 2) / 2]

--=Tabel Perbandingan Sebelum vs. Sesudah Pembersihan


,Indikator Kualitas Data,Sebelum Pembersihan,Sesudah Pembersihan
0,Total Baris,"2,010,000","1,984,994"
1,Order ID Unik,"2,000,000","1,984,994"
2,Duplikat Order ID,"10,000",0
3,Null pada qty,"6,173",0
4,Null pada harga_satuan,"5,930",0
5,Harga Satuan Negatif (< 0),"2,040",0
6,Outlier qty (> 12.0),969,0
7,Penulisan Kota Tidak Konsisten,"100,768",0
8,Jumlah Variasi String Kota,24,8


In [8]:
import time
from pyspark.sql.functions import count

print("menyimpan Parquet ke HDFS dan membandingkan dengan CSV\n")

path_mentah = f"{sumber_data}/raw/transaksi/"
path_bersih = f"{sumber_data}/curated/transaksi"

# 1. Simpan dataset bersih ke HDFS /tidarmart/curated/transaksi dalam format Parquet berpartisi 'bulan'
transaksi_curated.write.mode("overwrite").partitionBy("bulan").parquet(path_bersih)

print("--== Struktur Partisi Parquet di HDFS ==--")
!hdfs dfs -ls /tidarmart/curated/

print("\n--== Perbandingan Ukuran di HDFS: CSV Mentah vs. Parquet Bersih ==--")
!hdfs dfs -du -h /tidarmart/raw/transaksi
!hdfs dfs -du -h /tidarmart/curated/transaksi

# 2. Benchmark waktu eksekusi satu query agregasi yang sama:
#    Filter transaksi bulan 2024-06, lalu hitung jumlah transaksi dan total nilai per kota

# A. Eksekusi pada CSV mentah
mulai_csv = time.time()
uji_csv = spark.read.csv(path_mentah, header=True, schema=skema_transaksi)
hasil_csv = (
    uji_csv
    .filter(date_format(col("tanggal"), "yyyy-MM") == "2024-06")
    .groupBy("kota")
    .agg(
        count("order_id").alias("jumlah_transaksi"),
        spark_sum(round(col("qty") * col("harga_satuan") * (1 - col("diskon_persen") / 100.0), 2)).alias("total_nilai")
    )
    .collect()
)
waktu_csv = time.time() - mulai_csv

# B. Eksekusi pada Parquet berpartisi
mulai_parquet = time.time()
uji_parquet = spark.read.parquet(path_bersih)
hasil_parquet = (
    uji_parquet
    .filter(col("bulan") == "2024-06")
    .groupBy("kota")
    .agg(
        count("order_id").alias("jumlah_transaksi"),
        spark_sum("total_nilai").alias("total_nilai")
    )
    .collect()
)
waktu_parquet = time.time() - mulai_parquet

tabel_perbandingan = pd.DataFrame([
    {
        "Format Penyimpanan": "CSV Mentah (/tidarmart/raw/transaksi)",
        "Waktu Eksekusi Query (detik)": __builtins__.round(waktu_csv, 3)
    },
    {
        "Format Penyimpanan": "Parquet Berpartisi (/tidarmart/curated/transaksi)",
        "Waktu Eksekusi Query (detik)": __builtins__.round(waktu_parquet, 3)
    },
])

print(f"\n--=Tabel Perbandingan Waktu Eksekusi Query (Parquet {waktu_csv / waktu_parquet:.2f}x lebih cepat) ==--")
display(tabel_perbandingan)

menyimpan Parquet ke HDFS dan membandingkan dengan CSV



--== Struktur Partisi Parquet di HDFS ==--
Found 1 items
drwxr-xr-x   - cahyoalim supergroup          0 2026-11-25 13:42 /tidarmart/curated/transaksi

--== Perbandingan Ukuran di HDFS: CSV Mentah vs. Parquet Bersih ==--
46.8 M  46.8 M  /tidarmart/raw/transaksi/transaksi_part1.csv
46.8 M  46.8 M  /tidarmart/raw/transaksi/transaksi_part2.csv
46.8 M  46.8 M  /tidarmart/raw/transaksi/transaksi_part3.csv
46.8 M  46.8 M  /tidarmart/raw/transaksi/transaksi_part4.csv
0      0       /tidarmart/curated/transaksi/_SUCCESS
9.3 M  28.0 M  /tidarmart/curated/transaksi/bulan=2026-01
8.4 M  25.3 M  /tidarmart/curated/transaksi/bulan=2026-02
9.3 M  28.0 M  /tidarmart/curated/transaksi/bulan=2026-03
9.0 M  27.1 M  /tidarmart/curated/transaksi/bulan=2026-04
9.4 M  28.1 M  /tidarmart/curated/transaksi/bulan=2026-05
9.1 M  27.2 M  /tidarmart/curated/transaksi/bulan=2026-06


[Stage 84:>                                                         (0 + 2) / 2]


--=Tabel Perbandingan Waktu Eksekusi Query (Parquet 12.06x lebih cepat) ==--


,Format Penyimpanan,Waktu Eksekusi Query (detik)
0,CSV Mentah (/tidarmart/raw/transaksi),4.849
1,Parquet Berpartisi (/tidarmart/curated/transaksi),0.402


## Interpretasi Tahap 3

## Tahap 4 – Analitik Batch (DataFrame, Join, Window, Spark SQL)


In [10]:
from pyspark.sql.functions import (
    col, sum as spark_sum, count, round as spark_round, broadcast
)

#Baca data Parquet bersih dari HDFS & saring pesanan yang tidak dibatalkan (flag_batal == 0)
transaksi_parquet = spark.read.parquet(f"{sumber_data}/curated/transaksi")
transaksi_valid = transaksi_parquet.filter(col("flag_batal") == 0)

#Siapkan tabel dimensi (pilih kolom yang dibutuhkan agar tidak ada nama kolom ganda)
produk_dim = produk_mentah.select("produk_id", "nama_produk", "kategori", "harga_dasar")
pelanggan_dim = pelanggan_mentah.select(
    "pelanggan_id",
    "nama",
    "membership",
    col("kota").alias("kota_pelanggan")
)

#A: Join transaksi dengan produk & pelanggan memakai strategi Broadcast Join
data_gabungan = (
    transaksi_valid
    .join(broadcast(produk_dim), on="produk_id", how="inner")
    .join(broadcast(pelanggan_dim), on="pelanggan_id", how="inner")
)

#Hitung omzet per kategori per kota
hasil_A = (
    data_gabungan
    .groupBy("kota", "kategori")
    .agg(
        count("order_id").alias("jumlah_transaksi"),
        spark_round(spark_sum("total_nilai"), 2).alias("total_omzet")
    )
    .orderBy("kota", col("total_omzet").desc())
)

print("--== ANALISIS A: Omzet per Kategori per Kota (20 Baris Teratas) ==--")
hasil_A.show(20, truncate=False)

print("--== Bukti Strategi Join (BroadcastHashJoin) pada Rencana Eksekusi ==--")
hasil_A.explain()

--== ANALISIS A: Omzet per Kategori per Kota (20 Baris Teratas) ==--


[Stage 89:>                                                         (0 + 2) / 2]

+---------+------------+----------------+--------------+
|kota     |kategori    |jumlah_transaksi|total_omzet   |
+---------+------------+----------------+--------------+
|Kebumen  |Rumah Tangga|26113           |2.962345775E10|
|Kebumen  |Makanan     |25292           |2.962244575E10|
|Kebumen  |Kesehatan   |25830           |2.8114697E10  |
|Kebumen  |Elektronik  |24638           |2.79756095E10 |
|Kebumen  |Otomotif    |22788           |2.6932498E10  |
|Kebumen  |Buku        |22791           |2.6451826E10  |
|Kebumen  |Fashion     |24217           |2.633652625E10|
|Kebumen  |Olahraga    |22426           |2.3087448E10  |
|Magelang |Makanan     |26257           |3.14264675E10 |
|Magelang |Rumah Tangga|27543           |3.065897E10   |
|Magelang |Elektronik  |25764           |3.0099561E10  |
|Magelang |Kesehatan   |26937           |2.911627325E10|
|Magelang |Otomotif    |24155           |2.8847425E10  |
|Magelang |Buku        |23675           |2.761674825E10|
|Magelang |Fashion     |25356  

In [11]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

#Hitung total omzet dan jumlah transaksi per bulan
omzet_bulanan = (
    data_gabungan
    .groupBy("bulan")
    .agg(
        count("order_id").alias("jumlah_transaksi"),
        spark_round(spark_sum("total_nilai"), 2).alias("total_omzet")
    )
)

#Window diurutkan berdasarkan bulan untuk mengambil omzet bulan sebelumnya (lag)
window_bulan = Window.orderBy("bulan")

hasil_B = (
    omzet_bulanan
    .withColumn("omzet_bulan_lalu", lag("total_omzet", 1).over(window_bulan))
    .withColumn(
        "pertumbuhan_persen",
        spark_round(((col("total_omzet") - col("omzet_bulan_lalu")) / col("omzet_bulan_lalu")) * 100, 2)
    )
    .orderBy("bulan")
)

print("--== ANALISIS B: Tren Omzet Bulanan & Pertumbuhan MoM (%) ==--")
hasil_B.show(truncate=False)

--== ANALISIS B: Tren Omzet Bulanan & Pertumbuhan MoM (%) ==--
+-------+----------------+---------------+----------------+------------------+
|bulan  |jumlah_transaksi|total_omzet    |omzet_bulan_lalu|pertumbuhan_persen|
+-------+----------------+---------------+----------------+------------------+
|2026-01|272632          |3.058679785E11 |NULL            |NULL              |
|2026-02|246195          |2.773332895E11 |3.058679785E11  |-9.33             |
|2026-03|272629          |3.0542464575E11|2.773332895E11  |10.13             |
|2026-04|263885          |2.9681873775E11|3.0542464575E11 |-2.82             |
|2026-05|273606          |3.0728134675E11|2.9681873775E11 |3.52              |
|2026-06|263867          |2.97134213E11  |3.0728134675E11 |-3.3              |
+-------+----------------+---------------+----------------+------------------+



In [12]:
from pyspark.sql.functions import dense_rank

#Hitung total omzet tiap produk di masing-masing kota
omzet_produk_kota = (
    data_gabungan
    .groupBy("kota", "produk_id", "nama_produk", "kategori")
    .agg(
        count("order_id").alias("jumlah_transaksi"),
        spark_round(spark_sum("total_nilai"), 2).alias("total_omzet")
    )
)

#Window dipartisi per kota dan diurutkan dari omzet tertinggi
window_kota = Window.partitionBy("kota").orderBy(col("total_omzet").desc())

hasil_C = (
    omzet_produk_kota
    .withColumn("peringkat", dense_rank().over(window_kota))
    .filter(col("peringkat") <= 5)
    .orderBy("kota", "peringkat")
)

print("--== ANALISIS C: Peringkat 5 Produk Terlaris di Setiap Kota ==--")
hasil_C.show(40, truncate=False)

--== ANALISIS C: Peringkat 5 Produk Terlaris di Setiap Kota ==--


[Stage 102:============================>                            (1 + 1) / 2]

+----------+---------+-----------------+------------+----------------+-----------+---------+
|kota      |produk_id|nama_produk      |kategori    |jumlah_transaksi|total_omzet|peringkat|
+----------+---------+-----------------+------------+----------------+-----------+---------+
|Kebumen   |1007     |Buku-1007        |Buku        |65              |3.7935E8   |1        |
|Kebumen   |4361     |Buku-4361        |Buku        |56              |3.22725E8  |2        |
|Kebumen   |1002     |Rumah Tangga-1002|Rumah Tangga|57              |3.225E8    |3        |
|Kebumen   |3180     |Buku-3180        |Buku        |51              |3.12375E8  |4        |
|Kebumen   |2406     |Otomotif-2406    |Otomotif    |52              |3.09225E8  |5        |
|Magelang  |1600     |Elektronik-1600  |Elektronik  |63              |3.6195E8   |1        |
|Magelang  |2043     |Rumah Tangga-2043|Rumah Tangga|52              |3.30975E8  |2        |
|Magelang  |2686     |Makanan-2686     |Makanan     |54              |

In [15]:
from pyspark.sql.functions import to_date, avg

#Agregasi omzet harian per kategori 
omzet_harian_kategori = (
    data_gabungan
    .withColumn("tanggal_hari", to_date(col("tanggal")))
    .groupBy("kategori", "tanggal_hari")
    .agg(spark_round(spark_sum("total_nilai"), 2).alias("omzet_harian"))
)

# 2. Definisi Window dengan rentang baris (rowsBetween)
# - Kumulatif (running total): dari baris pertama kategori hingga baris saat ini
window_kumulatif = (
    Window.partitionBy("kategori")
    .orderBy("tanggal_hari")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# - Rata-rata bergerak 7 hari: 6 baris hari sebelumnya hingga hari saat ini (-6 sampai 0)
window_7_hari = (
    Window.partitionBy("kategori")
    .orderBy("tanggal_hari")
    .rowsBetween(-6, 0)
)

hasil_D = (
    omzet_harian_kategori
    .withColumn("omzet_kumulatif", spark_round(spark_sum("omzet_harian").over(window_kumulatif), 2))
    .withColumn("rata_rata_7_hari", spark_round(avg("omzet_harian").over(window_7_hari), 2))
    .orderBy("kategori", "tanggal_hari")
)

print("--== ANALISIS D: Omzet Kumulatif & Rata-Rata Bergerak 7 Hari (Contoh 15 Baris Pertama) ==--")
hasil_D.show(15, truncate=False)

--== ANALISIS D: Omzet Kumulatif & Rata-Rata Bergerak 7 Hari (Contoh 15 Baris Pertama) ==--


[Stage 156:>                                                        (0 + 2) / 2]

+--------+------------+------------+---------------+----------------+
|kategori|tanggal_hari|omzet_harian|omzet_kumulatif|rata_rata_7_hari|
+--------+------------+------------+---------------+----------------+
|Buku    |2026-01-01  |1.0997725E9 |1.0997725E9    |1.0997725E9     |
|Buku    |2026-01-02  |1.2248205E9 |2.324593E9     |1.1622965E9     |
|Buku    |2026-01-03  |1.35353375E9|3.67812675E9   |1.22604225E9    |
|Buku    |2026-01-04  |1.05648275E9|4.7346095E9    |1.183652375E9   |
|Buku    |2026-01-05  |1.15927325E9|5.89388275E9   |1.17877655E9    |
|Buku    |2026-01-06  |1.185608E9  |7.07949075E9   |1.179915125E9   |
|Buku    |2026-01-07  |1.27460575E9|8.3540965E9    |1.19344235714E9 |
|Buku    |2026-01-08  |1.31040625E9|9.66450275E9   |1.22353289286E9 |
|Buku    |2026-01-09  |1.16326E9   |1.082776275E10 |1.21473853571E9 |
|Buku    |2026-01-10  |1.1924745E9 |1.202023725E10 |1.19173007143E9 |
|Buku    |2026-01-11  |1.23242875E9|1.3252666E10   |1.21686521429E9 |
|Buku    |2026-01-12

In [14]:
# Daftarkan DataFrame gabungan sebagai Temporary View agar bisa di-query lewat Spark SQL
data_gabungan.createOrReplaceTempView("transaksi_gabungan_sql")

#Tulis ulang Analisis B (Tren Omzet Bulanan & Pertumbuhan MoM) menggunakan Spark SQL
hasil_E_sql = spark.sql("""
    WITH omzet_bulanan AS (
        SELECT
            bulan,
            COUNT(order_id) AS jumlah_transaksi,
            ROUND(SUM(total_nilai), 2) AS total_omzet
        FROM transaksi_gabungan_sql
        GROUP BY bulan
    ),
    dengan_lag AS (
        SELECT
            bulan,
            jumlah_transaksi,
            total_omzet,
            LAG(total_omzet, 1) OVER (ORDER BY bulan) AS omzet_bulan_lalu
        FROM omzet_bulanan
    )
    SELECT
        bulan,
        jumlah_transaksi,
        total_omzet,
        omzet_bulan_lalu,
        ROUND(((total_omzet - omzet_bulan_lalu) / omzet_bulan_lalu) * 100, 2) AS pertumbuhan_persen
    FROM dengan_lag
    ORDER BY bulan
""")

print("--== ANALISIS E: Hasil Analisis B Versi Spark SQL ==--")
hasil_E_sql.show(truncate=False)

# 3. Pembuktian kedua hasil identik (menggunakan subtract dua arah)
selisih_df_ke_sql = hasil_B.subtract(hasil_E_sql).count()
selisih_sql_ke_df = hasil_E_sql.subtract(hasil_B).count()
identik = (selisih_df_ke_sql == 0) and (selisih_sql_ke_df == 0) and (hasil_B.count() == hasil_E_sql.count())

print(f"Selisih (DataFrame - SQL): {selisih_df_ke_sql} baris")
print(f"Selisih (SQL - DataFrame): {selisih_sql_ke_df} baris")
print(f"Apakah hasil DataFrame API dan Spark SQL 100% identik? -> {identik}")

# 4. Perbandingan rencana eksekusi (Physical Plan)
print("\n--== Rencana Eksekusi (explain) Versi DataFrame API (Analisis B) ==--")
hasil_B.explain()

print("\n--== Rencana Eksekusi (explain) Versi Spark SQL (Analisis E) ==--")
hasil_E_sql.explain()

--== ANALISIS E: Hasil Analisis B Versi Spark SQL ==--
+-------+----------------+---------------+----------------+------------------+
|bulan  |jumlah_transaksi|total_omzet    |omzet_bulan_lalu|pertumbuhan_persen|
+-------+----------------+---------------+----------------+------------------+
|2026-01|272632          |3.058679785E11 |NULL            |NULL              |
|2026-02|246195          |2.773332895E11 |3.058679785E11  |-9.33             |
|2026-03|272629          |3.0542464575E11|2.773332895E11  |10.13             |
|2026-04|263885          |2.9681873775E11|3.0542464575E11 |-2.82             |
|2026-05|273606          |3.0728134675E11|2.9681873775E11 |3.52              |
|2026-06|263867          |2.97134213E11  |3.0728134675E11 |-3.3              |
+-------+----------------+---------------+----------------+------------------+

Selisih (DataFrame - SQL): 0 baris
Selisih (SQL - DataFrame): 0 baris
Apakah hasil DataFrame API dan Spark SQL 100% identik? -> True

--== Rencana Eksekus

In [16]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, max as spark_max, min as spark_min,
    round as spark_round, datediff, lit, row_number
)
from pyspark.sql.window import Window

# 1. Cari tanggal transaksi paling akhir di dataset sebagai acuan hitung Recency
tanggal_maks = transaksi_valid.agg(spark_max("tanggal")).collect()[0][0]
print(f"Tanggal acuan (terakhir di dataset): {tanggal_maks}")

# 2. Hitung metrik RFM per pelanggan
rfm_pelanggan = (
    transaksi_valid
    .groupBy("pelanggan_id")
    .agg(
        datediff(lit(tanggal_maks), spark_max("tanggal")).alias("recency_hari"),
        count("order_id").alias("frequency"),
        spark_round(spark_sum("total_nilai"), 2).alias("monetary")
    )
)

print("\n--== ANALISIS F: Contoh 10 Pelanggan dengan Monetary Tertinggi ==--")
rfm_pelanggan.orderBy(col("monetary").desc()).show(10, truncate=False)

print("--== Statistik Ringkas RFM Seluruh Pelanggan Aktif ==--")
rfm_pelanggan.select("recency_hari", "frequency", "monetary").summary("count", "mean", "min", "50%", "max").show()

# 3. Analisis Prinsip Pareto: Berapa persen pelanggan menyumbang 80% total omzet?
total_pelanggan_aktif = rfm_pelanggan.count()
total_omzet_semua = rfm_pelanggan.agg(spark_sum("monetary")).collect()[0][0]

# Urutkan pelanggan dari belanja terbesar ke terkecil, lalu hitung omzet kumulatif
window_pareto = (
    Window.orderBy(col("monetary").desc(), col("pelanggan_id"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
window_urut = Window.orderBy(col("monetary").desc(), col("pelanggan_id"))

pareto_df = (
    rfm_pelanggan
    .withColumn("urutan_pelanggan", row_number().over(window_urut))
    .withColumn("omzet_kumulatif", spark_sum("monetary").over(window_pareto))
    .withColumn("persen_omzet_kumulatif", spark_round((col("omzet_kumulatif") / lit(total_omzet_semua)) * 100, 4))
)

# Ambil urutan pelanggan pertama yang kontribusi kumulatifnya mencapai >= 80%
jumlah_pelanggan_80 = (
    pareto_df
    .filter(col("persen_omzet_kumulatif") >= 80.0)
    .agg(spark_min("urutan_pelanggan"))
    .collect()[0][0]
)
persen_pelanggan_80 = (jumlah_pelanggan_80 / total_pelanggan_aktif) * 100

print("--== Hasil Analisis Prinsip Pareto (80% Omzet) ==--")
print(f"Total Pelanggan Aktif          : {total_pelanggan_aktif:,} pelanggan")
print(f"Total Omzet Valid              : Rp {total_omzet_semua:,.2f}")
print(f"Jumlah Pelanggan Penyumbang 80%: {jumlah_pelanggan_80:,} pelanggan")
print(f"Persentase Pelanggan Pareto    : {persen_pelanggan_80:.2f}% pelanggan menyumbang 80% total omzet")

Tanggal acuan (terakhir di dataset): 2026-06-30 23:59:57

--== ANALISIS F: Contoh 10 Pelanggan dengan Monetary Tertinggi ==--


+------------+------------+---------+------------+
|pelanggan_id|recency_hari|frequency|monetary    |
+------------+------------+---------+------------+
|47055       |0           |2633     |3.1561525E9 |
|9678        |0           |1699     |1.87147925E9|
|18585       |0           |1632     |1.81242925E9|
|17840       |0           |1245     |1.40132625E9|
|10459       |0           |1188     |1.36792225E9|
|8642        |0           |1161     |1.34725825E9|
|30782       |0           |1146     |1.275548E9  |
|31562       |0           |1122     |1.22702875E9|
|24510       |0           |947      |1.186131E9  |
|30325       |0           |1006     |1.18481225E9|
+------------+------------+---------+------------+
only showing top 10 rows

--== Statistik Ringkas RFM Seluruh Pelanggan Aktif ==--
+-------+-----------------+-----------------+--------------------+
|summary|     recency_hari|        frequency|            monetary|
+-------+-----------------+-----------------+--------------------+
|  

In [17]:
from pyspark.sql.functions import avg, countDistinct, when, broadcast

# Gabungkan seluruh transaksi bersih (termasuk yang batal) dengan dimensi pelanggan
transaksi_membership = transaksi_parquet.join(broadcast(pelanggan_dim), on="pelanggan_id", how="inner")

hasil_G = (
    transaksi_membership
    .groupBy("membership")
    .agg(
        countDistinct("pelanggan_id").alias("jumlah_pelanggan"),
        count("order_id").alias("total_pesanan"),
        spark_round(count("order_id") / countDistinct("pelanggan_id"), 2).alias("rata_rata_frekuensi"),
        spark_round(avg(when(col("flag_batal") == 0, col("total_nilai"))), 2).alias("rata_rata_nilai_pesanan"),
        spark_round(avg("flag_batal") * 100, 2).alias("tingkat_pembatalan_persen")
    )
    .orderBy(col("rata_rata_nilai_pesanan").desc())
)

print("--== ANALISIS G: Perbandingan Perilaku Belanja Antar Membership ==--")
hasil_G.show(truncate=False)

--== ANALISIS G: Perbandingan Perilaku Belanja Antar Membership ==--


[Stage 193:>                                                        (0 + 2) / 2]

+----------+----------------+-------------+-------------------+-----------------------+-------------------------+
|membership|jumlah_pelanggan|total_pesanan|rata_rata_frekuensi|rata_rata_nilai_pesanan|tingkat_pembatalan_persen|
+----------+----------------+-------------+-------------------+-----------------------+-------------------------+
|Gold      |14901           |608757       |40.85              |1126064.06             |19.81                    |
|Silver    |29670           |1187139      |40.01              |1122671.4              |19.73                    |
|Platinum  |4806            |189098       |39.35              |1122654.3              |19.75                    |
+----------+----------------+-------------+-------------------+-----------------------+-------------------------+



In [18]:
# 1. Tandai status risiko pelanggan (Berisiko Pergi jika recency > 30 hari)
rfm_dengan_profil = (
    rfm_pelanggan
    .join(broadcast(pelanggan_dim), on="pelanggan_id", how="inner")
    .withColumn(
        "status_pelanggan",
        when(col("recency_hari") > 30, "Berisiko Pergi (>30 hari)").otherwise("Aktif (<=30 hari)")
    )
)

# 2. Perbandingan ringkas: Pelanggan Berisiko Pergi vs. Pelanggan Aktif
ringkasan_risiko = (
    rfm_dengan_profil
    .groupBy("status_pelanggan")
    .agg(
        count("pelanggan_id").alias("jumlah_pelanggan"),
        spark_round(count("pelanggan_id") / lit(total_pelanggan_aktif) * 100, 2).alias("persentase_pelanggan"),
        spark_round(avg("recency_hari"), 1).alias("rata_rata_recency_hari"),
        spark_round(avg("frequency"), 2).alias("rata_rata_frequency"),
        spark_round(avg("monetary"), 2).alias("rata_rata_monetary")
    )
)

print("--== ANALISIS H: Ringkasan Pelanggan Berisiko Pergi vs. Aktif ==--")
ringkasan_risiko.show(truncate=False)

# 3. Bedah profil khusus pelanggan yang berisiko pergi
pelanggan_berisiko = rfm_dengan_profil.filter(col("recency_hari") > 30)
total_berisiko = pelanggan_berisiko.count()

print(f"--== Profil Pelanggan Berisiko Pergi ({total_berisiko:,} pelanggan) Berdasarkan Membership ==--")
(
    pelanggan_berisiko
    .groupBy("membership")
    .agg(
        count("pelanggan_id").alias("jumlah_berisiko"),
        spark_round(count("pelanggan_id") / lit(total_berisiko) * 100, 2).alias("persen_dari_berisiko"),
        spark_round(avg("frequency"), 2).alias("rata_frequency"),
        spark_round(avg("monetary"), 2).alias("rata_monetary")
    )
    .orderBy(col("jumlah_berisiko").desc())
    .show(truncate=False)
)

print("--== Profil Pelanggan Berisiko Pergi Berdasarkan Kota ==--")
(
    pelanggan_berisiko
    .groupBy("kota_pelanggan")
    .agg(
        count("pelanggan_id").alias("jumlah_berisiko"),
        spark_round(count("pelanggan_id") / lit(total_berisiko) * 100, 2).alias("persen_dari_berisiko"),
        spark_round(avg("monetary"), 2).alias("rata_monetary")
    )
    .orderBy(col("jumlah_berisiko").desc())
    .show(truncate=False)
)

--== ANALISIS H: Ringkasan Pelanggan Berisiko Pergi vs. Aktif ==--
+-------------------------+----------------+--------------------+----------------------+-------------------+------------------+
|status_pelanggan         |jumlah_pelanggan|persentase_pelanggan|rata_rata_recency_hari|rata_rata_frequency|rata_rata_monetary|
+-------------------------+----------------+--------------------+----------------------+-------------------+------------------+
|Aktif (<=30 hari)        |40918           |83.35               |7.3                   |37.91              |4.260723635E7     |
|Berisiko Pergi (>30 hari)|8174            |16.65               |66.4                  |5.09               |5683547.13        |
+-------------------------+----------------+--------------------+----------------------+-------------------+------------------+

--== Profil Pelanggan Berisiko Pergi (8,174 pelanggan) Berdasarkan Membership ==--


+----------+---------------+--------------------+--------------+-------------+
|membership|jumlah_berisiko|persen_dari_berisiko|rata_frequency|rata_monetary|
+----------+---------------+--------------------+--------------+-------------+
|Silver    |4935           |60.37               |5.03          |5714671.83   |
|Gold      |2416           |29.56               |5.16          |5722288.39   |
|Platinum  |823            |10.07               |5.22          |5383183.48   |
+----------+---------------+--------------------+--------------+-------------+

--== Profil Pelanggan Berisiko Pergi Berdasarkan Kota ==--
+--------------+---------------+--------------------+-------------+
|kota_pelanggan|jumlah_berisiko|persen_dari_berisiko|rata_monetary|
+--------------+---------------+--------------------+-------------+
|Purworejo     |1071           |13.1                |5750918.77   |
|Wonosobo      |1062           |12.99               |5538710.22   |
|Semarang      |1019           |12.47          